# FarmTech Solutions — Fase 6: visão computacional
**Autores:** Kauan Maciel Forgiarini (RM 574005) e Wagner Adriano de Souza Silva Junior (RM 569431) | **Grupo 9**

## Estado desta versão
Implementação preparada, **ainda não executada com dados reais**. Não contém resultados experimentais inventados. Antes da entrega: preparar e rotular as imagens, executar todas as células no Colab, interpretar as saídas e inserir os links públicos do notebook e do vídeo.

## 1. Problema e objetivo
Como integrante do time de IA da FarmTech Solutions, proponho uma demonstração de detecção de **pessoas e carros** em acessos a propriedades. O objetivo é comparar um YOLO personalizado com 30 e 60 épocas, um YOLO pré-treinado sem adaptação e uma CNN construída do zero.

Este é um protótipo acadêmico: detectar uma pessoa não identifica quem ela é nem autoriza concluir que existe invasão. O conjunto mínimo não sustenta uso em produção.

### Escopo e escolha técnica
As Entregas 1 e 2 são obrigatórias pelo barema. As opções “Ir Além” não são implementadas aqui. Utilizamos YOLOv8n pela API Ultralytics. O enunciado menciona diretórios do YOLOv5 como exemplo; nesta implementação os resultados ficam em `resultados/`. Interpretamos “YOLO padrão” como os pesos COCO sem treinamento adicional. **O capítulo 3 não foi fornecido: conferir se a disciplina exige uma implementação específica antes da entrega.**


## Continuidade com a Fase 5

A fase anterior analisou condições climáticas e produtividade agrícola com clusterização e modelos de regressão. Nesta fase, a FarmTech amplia sua atuação para visão computacional. O dataset tabular `crop_yield.csv` não substitui as imagens exigidas aqui. Mantemos a organização documental (README introdutório e notebook analítico), mas desenvolvemos uma solução independente em novo repositório.

Referência do trabalho anterior: https://github.com/KauanForgiarini/farmtech-fase5-ml


## 2. Dataset e rotulação no Make Sense AI
1. Obter 40 fotografias de pessoas e 40 de carros, próprias ou com licença que permita uso e compartilhamento. Evitar imagens sensíveis e dados pessoais identificáveis.
2. Escolher imagens contendo somente uma das duas classes-alvo (várias instâncias dessa mesma classe são permitidas). Variar fundo, iluminação, distância e ângulo. Não usar cópias, recortes ou quadros vizinhos da mesma cena em divisões diferentes.
3. Separar **32 treino + 4 validação + 4 teste por classe**, antes de qualquer aumento de dados. Total: 64/8/8.
4. Em https://www.makesense.ai/, criar os rótulos nesta ordem: **pessoa (0), carro (1)**. Desenhar caixas ajustadas em todas as instâncias-alvo e exportar no formato YOLO. Rotular também validação e teste para medir detecção quantitativamente.
5. No Drive criar `MyDrive/FarmTech_Fase6/dataset/images/{train,val,test}` e `dataset/labels/{train,val,test}`. Cada imagem precisa de um `.txt` de mesmo nome. Por exemplo, `images/train/pessoa_001.jpg` e `labels/train/pessoa_001.txt`.
6. Preencher `fontes.csv` com origem, licença e identificação da cena. Manter cada cena em uma única divisão. Compartilhar o dataset com o avaliador; cada pessoa pode copiar a pasta para seu próprio Drive.

Formato de cada linha do TXT: `classe centro_x centro_y largura altura`, coordenadas normalizadas entre 0 e 1. O ZIP exportado pelo Make Sense precisa ser extraído e os TXT distribuídos entre as pastas correspondentes.

**Critério de seleção pré-definido:** escolher entre os dois modelos personalizados pelo maior mAP50–95 na validação. Usar teste apenas na avaliação final, sem ajustar parâmetros a partir dele.


In [ ]:
# Instalação no Colab; o ambiente efetivamente usado será registrado.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics>=8.3,<9', 'scikit-learn', 'pandas', 'pyyaml', 'matplotlib'], check=True)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Caminhos, sementes e bibliotecas comuns aos experimentos.
from pathlib import Path
import random, json, hashlib, time, copy
import numpy as np
import pandas as pd
import yaml
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, ConfusionMatrixDisplay
from ultralytics import YOLO

BASE = Path('/content/drive/MyDrive/FarmTech_Fase6')
DATA = BASE / 'dataset'
OUT = BASE / 'resultados'
OUT.mkdir(parents=True, exist_ok=True)
CLASSES = ['pessoa', 'carro']
SEED = 42
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
YOLO_DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all()
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(OUT / 'requirements-executado.txt').write_text(freeze)
(OUT / 'ambiente.json').write_text(json.dumps({
    'python': sys.version, 'torch': torch.__version__, 'device': DEVICE,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seed': SEED}, indent=2))
print('Dispositivo:', DEVICE)


In [ ]:
# Verificação dos arquivos, caixas, classes, duplicatas exatas e proporções.
# A revisão humana de cenas semelhantes continua necessária.
rows, seen = [], {}
for split, required in [('train', 32), ('val', 4), ('test', 4)]:
    paths = sorted(p for p in (DATA / 'images' / split).glob('*')
                   if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    assert paths, f'Não há imagens em {split}. Prepare o dataset antes de continuar.'
    assert len({p.stem for p in paths}) == len(paths), 'Nomes-base repetidos.'
    for p in paths:
        with Image.open(p) as im:
            rgb = im.convert('RGB')
            digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
        assert digest not in seen, f'Imagem duplicada: {p} e {seen.get(digest)}'
        seen[digest] = str(p)
        label = DATA / 'labels' / split / (p.stem + '.txt')
        assert label.exists(), f'Rotulação ausente: {label}'
        boxes = [list(map(float, line.split())) for line in label.read_text().splitlines() if line.strip()]
        assert boxes, f'Nenhuma caixa em {label}'
        for b in boxes:
            assert len(b) == 5 and np.isfinite(b).all(), f'Linha inválida em {label}'
            c, x, y, w, h = b
            assert c in [0, 1] and 0 < w <= 1 and 0 < h <= 1
            assert 0 <= x <= 1 and 0 <= y <= 1
            assert x-w/2 >= -0.001 and x+w/2 <= 1.001
            assert y-h/2 >= -0.001 and y+h/2 <= 1.001
        ids = {int(b[0]) for b in boxes}
        assert len(ids) == 1, 'Este protocolo de classificação exige uma classe-alvo por imagem.'
        rows.append({'path': str(p), 'split': split, 'class_id': ids.pop(), 'sha256': digest})
    counts = pd.DataFrame(rows).query('split == @split')['class_id'].value_counts()
    assert all(counts.get(i, 0) == required for i in range(2)), f'Esperado {required} por classe em {split}.'

manifest = pd.DataFrame(rows)
manifest.to_csv(OUT / 'manifesto.csv', index=False)
config = {'path': str(DATA), 'train': 'images/train', 'val': 'images/val',
          'test': 'images/test', 'names': dict(enumerate(CLASSES))}
YAML_PATH = BASE / 'dataset.yaml'
YAML_PATH.write_text(yaml.safe_dump(config, allow_unicode=True))
display(pd.crosstab(manifest['split'], manifest['class_id']).rename(columns=dict(enumerate(CLASSES))))


## 3. Entrega 1 — YOLO personalizado
Cada experimento parte dos **mesmos pesos pré-treinados**, sem continuar o anterior. Mantemos divisão, resolução, lote, semente e otimizador. Desativamos parada antecipada para cumprir 30 e 60 épocas e fixamos o fechamento do mosaic em zero para evitar uma mudança adicional de calendário.

O treinamento usa somente treino; a validação acompanha o ajuste e seleciona os pesos. Em detecção, **precisão, recall e mAP não são sinônimos de acurácia**. As perdas de caixa, classificação e DFL serão visualizadas separadamente. A CNN terá outra função de perda, portanto seus valores não devem ser comparados diretamente aos do YOLO.


In [ ]:
# Dois treinamentos independentes e avaliação na validação.
custom, durations, validation_rows = {}, {}, []
for epochs in [30, 60]:
    seed_all()
    model = YOLO('yolov8n.pt')
    start = time.perf_counter()
    model.train(data=str(YAML_PATH), epochs=epochs, imgsz=640, batch=8,
                device=YOLO_DEVICE, seed=SEED, deterministic=True, workers=0,
                optimizer='SGD', lr0=0.01, lrf=0.01, cos_lr=False,
                patience=0, close_mosaic=0, project=str(OUT / 'treinos'),
                name=f'yolo_{epochs}', exist_ok=False, plots=True)
    durations[epochs] = time.perf_counter() - start
    run_dir = Path(model.trainer.save_dir)
    best = YOLO(str(run_dir / 'weights' / 'best.pt'))
    custom[epochs] = best
    metrics = best.val(data=str(YAML_PATH), split='val', device=YOLO_DEVICE,
                       workers=0, project=str(OUT / 'validacao'), name=f'yolo_{epochs}')
    validation_rows.append({'epocas': epochs, 'precision': metrics.box.mp,
                            'recall': metrics.box.mr, 'mAP50': metrics.box.map50,
                            'mAP50_95': metrics.box.map, 'treino_s': durations[epochs]})
    history = pd.read_csv(run_dir / 'results.csv')
    history.columns = history.columns.str.strip()
    history.to_csv(OUT / f'historico_yolo_{epochs}.csv', index=False)
    display(Image.open(run_dir / 'results.png'))

validation = pd.DataFrame(validation_rows)
validation.to_csv(OUT / 'comparacao_validacao.csv', index=False)
chosen = int(validation.sort_values(['mAP50_95', 'epocas'], ascending=[False, True]).iloc[0]['epocas'])
display(validation)
print('Épocas selecionadas apenas pela validação:', chosen)


## 4. Entrega 2 — CNN do zero
A CNN recebe a imagem inteira em 128×128, usa três blocos convolucionais, pooling, dropout e saída para duas classes. Nenhum peso pré-treinado é carregado. O aumento de dados é aplicado somente ao treino. Salvamos a época de menor perda de validação entre 60 épocas.

Esse modelo responde “qual classe aparece na imagem?” e não produz caixas. Por isso a comparação comum usará acurácia por imagem; as métricas de localização do YOLO ficam separadas.


In [ ]:
# Dataset em memória pequena: 80 imagens, sem depender de pastas de classificação.
class ImageDataset(Dataset):
    def __init__(self, split, augment=False):
        self.rows = manifest[manifest.split == split].reset_index(drop=True)
        self.augment = augment
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        with Image.open(row.path) as source:
            im = source.convert('RGB').resize((128, 128))
        if self.augment and random.random() < 0.5:
            im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        x = torch.from_numpy(np.asarray(im, dtype=np.float32).copy() / 255).permute(2, 0, 1)
        return x, int(row.class_id)

seed_all()
cnn = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 2)
).to(DEVICE)
loaders = {s: DataLoader(ImageDataset(s, s == 'train'), batch_size=8,
                         shuffle=s == 'train', num_workers=0) for s in ['train', 'val']}
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001, weight_decay=0.0001)
criterion = nn.CrossEntropyLoss()
cnn_history, best_loss, best_state = [], float('inf'), None
start = time.perf_counter()
for epoch in range(60):
    record = {'epoch': epoch + 1}
    for split in ['train', 'val']:
        cnn.train(split == 'train')
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in loaders[split]:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.set_grad_enabled(split == 'train'):
                logits = cnn(x)
                loss = criterion(logits, y)
                if split == 'train':
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)
        record[f'{split}_loss'] = loss_sum / total
        record[f'{split}_accuracy'] = correct / total
    if record['val_loss'] < best_loss:
        best_loss = record['val_loss']
        best_state = copy.deepcopy(cnn.state_dict())
        best_epoch = epoch + 1
    cnn_history.append(record)
cnn_seconds = time.perf_counter() - start
cnn.load_state_dict(best_state)
cnn.eval()
torch.save({'state_dict': best_state, 'classes': CLASSES, 'epoch': best_epoch}, OUT / 'cnn_best.pt')
cnn_history = pd.DataFrame(cnn_history)
cnn_history.to_csv(OUT / 'historico_cnn.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cnn_history.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0], title='Perda CNN')
cnn_history.plot(x='epoch', y=['train_accuracy', 'val_accuracy'], ax=axes[1], title='Acurácia CNN')
plt.tight_layout()
plt.savefig(OUT / 'curvas_cnn.png')
plt.show()
print('Melhor época CNN pela validação:', best_epoch)


## 5. Teste final e comparação justa por imagem
A partir deste ponto, não usar os resultados para escolher hiperparâmetros. O teste contém somente oito imagens: cada erro altera a acurácia em **12,5 pontos percentuais**.

No YOLO padrão COCO, pessoa corresponde à classe 0 e carro à classe 2. No personalizado, os IDs são 0 e 1. Entre as detecções-alvo com confiança ≥0,25, usamos a de maior confiança; sem detecção contamos erro (abstenção), sem excluir a imagem. O limiar foi definido antes de olhar o teste.

Medimos o tempo ponta a ponta por imagem com aquecimento, sincronização CUDA e cinco repetições, incluindo pré-processamento e excluindo leitura de disco. Todos os modelos usam o mesmo dispositivo e imagens em memória; as resoluções nativas diferem (640 no YOLO e 128 na CNN), portanto o tempo compara estes pipelines, não apenas arquiteturas isoladas.


In [ ]:
# Inferência com mapeamento explícito das classes e temporização uniforme.
test_rows = manifest[manifest.split == 'test'].reset_index(drop=True)
test_images = [Image.open(p).convert('RGB') for p in test_rows.path]
y_true = test_rows.class_id.to_numpy()
baseline = YOLO('yolov8n.pt')

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def predict_yolo(model, im, standard=False):
    allowed = [0, 2] if standard else [0, 1]
    result = model.predict(im, imgsz=640, conf=0.25, classes=allowed,
                           device=YOLO_DEVICE, verbose=False)[0]
    if len(result.boxes) == 0:
        return -1
    index = int(result.boxes.conf.argmax().item())
    cls = int(result.boxes.cls[index].item())
    return {0: 0, 2: 1}[cls] if standard else cls

def predict_cnn(im):
    x = torch.from_numpy(np.asarray(im.resize((128, 128)), dtype=np.float32).copy() / 255)
    x = x.permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        return int(cnn(x).argmax(1).item())

predictors = {
    'YOLO padrão': lambda im: predict_yolo(baseline, im, True),
    'YOLO 30': lambda im: predict_yolo(custom[30], im),
    'YOLO 60': lambda im: predict_yolo(custom[60], im),
    'CNN do zero': predict_cnn
}
train_times = {'YOLO padrão': 0, 'YOLO 30': durations[30], 'YOLO 60': durations[60], 'CNN do zero': cnn_seconds}
comparison, predictions = [], {}
for name, predict in predictors.items():
    for _ in range(3):
        predict(test_images[0])
    pred, timings = [], []
    for im in test_images:
        for repeat in range(5):
            sync()
            start = time.perf_counter()
            value = predict(im)
            sync()
            timings.append((time.perf_counter() - start) * 1000)
        pred.append(value)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, labels=[0, 1], average='macro', zero_division=0)
    acc = accuracy_score(y_true, pred)
    predictions[name] = pred
    comparison.append({'modelo': name, 'acuracia_imagem': acc, 'erro_imagem': 1-acc,
                       'precisao_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
                       'abstencoes': pred.count(-1), 'treino_s': train_times[name],
                       'inferencia_mediana_ms': np.median(timings)})
    ConfusionMatrixDisplay.from_predictions(y_true, pred, labels=[0, 1, -1],
        display_labels=['pessoa', 'carro', 'sem detecção'], colorbar=False)
    plt.title(name)
    plt.savefig(OUT / (name.replace(' ', '_') + '_confusao.png'), bbox_inches='tight')
    plt.show()
comparison = pd.DataFrame(comparison)
comparison.to_csv(OUT / 'comparacao_teste.csv', index=False)
pd.DataFrame({'imagem': test_rows.path, 'classe_real': y_true, **predictions}).to_csv(OUT / 'predicoes.csv', index=False)
display(comparison)


In [ ]:
# Métricas de localização dos dois modelos personalizados e evidências visuais.
# O modelo padrão é comparado acima pela tarefa comum de classificação por imagem.
detection_rows = []
for epochs, model in custom.items():
    metrics = model.val(data=str(YAML_PATH), split='test', device=YOLO_DEVICE,
                       workers=0, project=str(OUT / 'teste_deteccao'), name=f'yolo_{epochs}')
    detection_rows.append({'epocas': epochs, 'precision': metrics.box.mp,
                           'recall': metrics.box.mr, 'mAP50': metrics.box.map50,
                           'mAP50_95': metrics.box.map})
detection = pd.DataFrame(detection_rows)
detection.to_csv(OUT / 'deteccao_teste.csv', index=False)
display(detection)

# Salvar todas as oito imagens processadas pelo modelo escolhido na validação.
preview_dir = OUT / 'prints_teste'
preview_dir.mkdir(exist_ok=True)
for index, im in enumerate(test_images):
    result = custom[chosen].predict(im, conf=0.25, imgsz=640,
                                   device=YOLO_DEVICE, verbose=False)[0]
    rendered = Image.fromarray(result.plot()[..., ::-1])
    rendered.save(preview_dir / f'teste_{index+1:02}.jpg')
    display(rendered)


## 6. Discussão crítica e conclusões
A célula seguinte escreve apenas fatos calculados. Depois da execução, completar a interpretação com referências aos gráficos e às imagens, inclusive erros e abstenções.

**Facilidade de integração:** o YOLO padrão exige carregar os pesos e inferir, mas depende das classes conhecidas. A personalização mantém a interface e acrescenta coleta, rotulação e treinamento. A CNN é pequena e foi implementada diretamente em PyTorch, mas exige pré-processamento próprio e não localiza objetos.

**Limitações:** 80 imagens são insuficientes para representar todos os cenários; oito testes produzem estimativas instáveis. Pesos COCO podem já ter visto fotografias disponíveis publicamente, por isso imagens próprias são preferíveis. Diferenças de desempenho não provam superioridade geral. Fundo e condições de captura podem produzir atalhos de classificação. Uma semente não mede variabilidade entre treinamentos. Ausência de detecção não equivale à ausência física de pessoa ou veículo.

**Preencher depois de executar:**
- O que mudou nas perdas de treino e validação entre 30 e 60 épocas? Há sinais de sobreajuste?
- Qual abordagem teve melhor acurácia e qual teve menor latência? Quantos acertos reais em oito testes?
- Quais imagens falharam e quais condições explicam os erros? Evitar atribuir causas sem evidência.
- O ganho da personalização compensa o tempo e a rotulação neste cenário?
- Qual próximo experimento aumentaria a confiança no resultado?


In [ ]:
# Resumo factual gerado somente após experimentos concluídos.
best_common = comparison.sort_values('acuracia_imagem', ascending=False).iloc[0]
summary = (
    f'O modelo personalizado selecionado pela validação foi o de {chosen} épocas. '
    f'A maior acurácia por imagem observada no teste foi {best_common.acuracia_imagem:.1%}, '
    f'obtida por {best_common.modelo} (pode haver empate). '
    f'O conjunto de teste contém {len(y_true)} imagens e cada erro altera a acurácia em '
    f'{100/len(y_true):.1f} pontos percentuais. '
    'Estes números descrevem somente esta amostra; não demonstram prontidão para produção.'
)
display(Markdown(summary))
(OUT / 'resumo_factual.txt').write_text(summary, encoding='utf-8')


## 7. Referências e entrega
- Enunciado FIAP: Fase 6, “O começo da rede neural”, prazo 13/10/2026 às 23h59.
- [Ultralytics YOLOv8](https://docs.ultralytics.com/models/yolov8/)
- [Treinamento Ultralytics](https://docs.ultralytics.com/modes/train/)
- [Validação Ultralytics](https://docs.ultralytics.com/modes/val/)
- [Make Sense AI](https://www.makesense.ai/)

Salvar este notebook **com as saídas**, preencher a análise crítica, publicar em novo repositório público Grupo 9, inserir links do Colab/dataset e do vídeo não listado no README. Confirmar acesso pelo avaliador. Enviar o link no portal e não realizar commits após a entrega.
